In [1]:
from __future__ import print_function
import argparse
import os
import shutil
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
from torchvision import transforms
import utils
import YeNet


In [2]:
seed = 42
torch.manual_seed(seed)
batch_size = 32
batch_norm = True
learning_rate = 1e-4
log_interval = 10
epoch_num = 10

train_transform = transforms.Compose([
    utils.RandomRot(),
    utils.RandomFlip(),
    utils.ToTensor()
])

train_cover_dir = r'..\..\..\images\BOSSbase\YeNet\train_cover'
train_stego_dir = r'..\..\..\images\BOSSbase\YeNet\train_stego'

print("Generate loaders...")
train_loader = utils.DataLoaderStego(
    train_cover_dir, 
    train_stego_dir,
    embedding_otf=False, 
    shuffle=True,
    pair_constraint=not(batch_norm),
    batch_size=batch_size, 
    transform=train_transform,
    num_workers=8, 
    pin_memory=False
)

valid_transform = transforms.Compose([
    utils.ToTensor()
])

valid_cover_dir = r'..\..\..\images\BOSSbase\YeNet\valid_cover'
valid_stego_dir = r'..\..\..\images\BOSSbase\YeNet\valid_stego'

valid_loader = utils.DataLoaderStego(
    valid_cover_dir, 
    valid_stego_dir,
    embedding_otf=False, 
    shuffle=False,
    pair_constraint=True, 
    batch_size=batch_size,
    transform=valid_transform,
    num_workers=8,
    pin_memory=False
)

print('train_loader have {} iterations, valid_loader have {} iterations'.format(
    len(train_loader), len(valid_loader)))
print("Generate model")
net = YeNet.YeNet(with_bn=batch_norm)
print("Generate loss and optimizer")

criterion = nn.CrossEntropyLoss().cuda()
optimizer = optim.Adadelta(
    net.parameters(), 
    lr=learning_rate, 
    rho=0.95, 
    eps=1e-8,
    weight_decay=5e-4
)

_time = time.time()
    
def train(epoch):
    net.train()
    running_loss = 0.
    running_accuracy = 0.
    for batch_idx, data in enumerate(train_loader):
        images, labels = Variable(data['images']), Variable(data['labels'])
        optimizer.zero_grad()
        outputs = net(images)
        accuracy = YeNet.accuracy(outputs, labels).data[0]
        running_accuracy += accuracy
        loss = criterion(outputs, labels)
        running_loss += loss.data[0]
        loss.backward()
        optimizer.step()
        if (batch_idx + 1) % log_interval == 0:
            running_accuracy /= log_interval
            running_loss /= log_interval
            print(('\nTrain epoch: {} [{}/{}]\tAccuracy: ' +
                  '{:.2f}%\tLoss: {:.6f}').format(
                  epoch, batch_idx + 1, len(train_loader), 
                  100 * running_accuracy, running_loss))
            running_loss = 0.
            running_accuracy = 0.
            net.train()

def valid():
    net.eval()
    valid_loss = 0.
    valid_accuracy = 0.
    correct = 0
    for data in valid_loader:
        # break
        images, labels = Variable(data['images']), Variable(data['labels'])
        outputs = net(images)
        valid_loss += criterion(outputs, labels).data[0]
        valid_accuracy += YeNet.accuracy(outputs, labels).data[0]
    valid_loss /= len(valid_loader)
    valid_accuracy /= len(valid_loader)
    print('\nTest set: Loss: {:.4f}, Accuracy: {:.2f}%)\n'.format(
        valid_loss, 100 * valid_accuracy))
    return valid_loss, valid_accuracy
    
def save_checkpoint(state, is_best, filename='checkpoints/checkpoint.pth.tar'):
    torch.save(state, filename)
    if is_best:
        shutil.copyfile(filename, 'checkpoints/model_best.pth.tar')
    
best_accuracy = 0.
for epoch in range(1, epoch_num + 1):
    print("Epoch:", epoch)
    print("Train")
    train(epoch)
    print("Time:", time.time() - _time)
    print("Test")
    _, accuracy = valid()
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        is_best = True
    else:
        is_best = False
    print("Time:", time.time() - _time)
    save_checkpoint({
        'epoch': epoch,
        'state_dict': net.state_dict(),
        'best_prec1': accuracy,
        'optimizer': optimizer.state_dict(),
    }, is_best)

Generate loaders...
train_loader have 64 iterations, valid_loader have 8 iterations
Generate model
Generate loss and optimizer
Epoch: 1
Train


c:\Users\Администратор\OneDrive\Рабочий стол\diplom\venv\Lib\site-packages\torch\utils\data\dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
c:\Users\Администратор\OneDrive\Рабочий стол\diplom\algorithms\steganalysis\YeNet-Pytorch\YeNet.py:63: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  nn.init.xavier_uniform(self.conv.weight)
c:\Users\Администратор\OneDrive\Рабочий стол\diplom\algorithms\steganalysis\YeNet-Pytorch\YeNet.py:121: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  nn.init.normal(mod.

FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\Users\Администратор\OneDrive\Рабочий стол\diplom\venv\Lib\site-packages\torch\utils\data\_utils\worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Администратор\OneDrive\Рабочий стол\diplom\venv\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "c:\Users\Администратор\OneDrive\Рабочий стол\diplom\algorithms\steganalysis\YeNet-Pytorch\utils.py", line 38, in __getitem__
    images = Image.open(cover_path)
             ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Администратор\OneDrive\Рабочий стол\diplom\venv\Lib\site-packages\PIL\Image.py", line 3512, in open
    fp = builtins.open(filename, "rb")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
File